In [ ]:
import os
import pandas as pd
import numpy as np

# weighting method 비교: CUE(softmax, 학습된 modality 가중치) vs FP-only vs IMG-only vs
# UNIFORM(고정 0.5/0.5) vs Random(Dirichlet(1,1)에서 1회 뽑아 고정한 가중치, 서로 다른 seed로 N회 반복).
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# Random 시도 횟수: cue/scripts/run_cue_random_*.sh가 R1..R{NUM_RANDOM_TRIALS} 폴더를 생성했다.
# 필요하면 이 값을 바꿔서(N을 줄여서) 앞쪽 N개 시행만 사용할 수 있다.
NUM_RANDOM_TRIALS = 30

# 고정(단일 결과) 모델 소스. CUE는 표/그래프에서 항상 마지막에 오도록 순서를 맞춘다.
model_sources = {
    'FP': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'FP'),
    'IMG': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'IMG'),
    'UNIFORM': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'UNIFORM'),
    'CUE': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE')
}
# Random 소스: {trial}을 R1..R{NUM_RANDOM_TRIALS}로 치환해 사용
random_template = os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'R{trial}')

# rmse는 낮을수록, r2/pcc는 높을수록 좋은 성능
LOWER_IS_BETTER = {'rmse': True, 'r2': False, 'pcc': False}

In [2]:
def load_metric(model_dir_template, dataset, target):
    """cmfe의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


# CUE / FP / IMG / UNIFORM (고정 모델) 로드
raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_fixed = pd.DataFrame(raw_rows)

# Random (R1..R{NUM_RANDOM_TRIALS}) 로드: 시행별로 개별 보관한다 (평균내지 않음 -> p-value 계산에 필요).
raw_rows_random = []
for d_name in datasets:
    for target in targets:
        for trial in range(1, NUM_RANDOM_TRIALS + 1):
            dir_template = random_template.replace('{trial}', str(trial))
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows_random.append({'Dataset': d_name, 'Target': target, 'Trial': trial, **vals})

df_random = pd.DataFrame(raw_rows_random)

print(f"✅ 고정 모델(CUE/FP/IMG/UNIFORM) 데이터 로드 완료: {len(df_fixed)}개")
print(f"✅ Random 시행 데이터 로드 완료: {len(df_random)}개 (기대값: {len(datasets)*len(targets)*NUM_RANDOM_TRIALS})")

✅ 고정 모델(CUE/FP/IMG/UNIFORM) 데이터 로드 완료: 32개
✅ Random 시행 데이터 로드 완료: 240개 (기대값: 240)


In [3]:
def build_ranked_table(df_all, dataset, target, metrics=metrics_of_interest, model_order=('FP', 'IMG', 'UNIFORM', 'Random', 'CUE')):
    """FP/IMG/UNIFORM/CUE는 고정값을, Random은 시행 평균을 대표값으로 사용해 avg_rank 포함 표를 만든다.
    CUE가 항상 마지막 행에 오도록 model_order 기본값을 고정한다.

    Returns:
        Optional[pd.DataFrame]: 컬럼이
            Model | avg_rank | rmse | rmse_rank | r2 | r2_rank | pcc | pcc_rank
            순서인 표. 성능 값은 소수점 셋째 자리, 순위는 소수점 첫째 자리로 반올림한다.
    """
    df_sub = df_all[(df_all['Dataset'] == dataset) & (df_all['Target'] == target)]
    df_sub = df_sub.set_index('Model').reindex(model_order)
    df_sub = df_sub.dropna(subset=[m for m in metrics if m in df_sub.columns], how='all')
    if df_sub.empty:
        return None

    table = pd.DataFrame(index=df_sub.index)
    rank_cols = []
    for m in metrics:
        if m not in df_sub.columns:
            continue
        table[m] = df_sub[m].round(3)
        rank_col = f'{m}_rank'
        table[rank_col] = df_sub[m].rank(ascending=LOWER_IS_BETTER.get(m, True), method='min').round(1)
        rank_cols.append(rank_col)

    table['avg_rank'] = table[rank_cols].mean(axis=1).round(1)

    ordered_cols = ['avg_rank']
    for m in metrics:
        if m in table.columns:
            ordered_cols += [m, f'{m}_rank']
    table = table[ordered_cols]

    return table.reset_index()


def build_pvalue_table(df_random, df_fixed, dataset, target, metrics=metrics_of_interest):
    """metric별로 'Random 시행이 CUE보다 더 좋았던 비율'을 계산한다.
    - rmse: 낮을수록 좋음 -> Random_trial < CUE
    - r2/pcc: 높을수록 좋음 -> Random_trial > CUE
    p_value = n_better / n_total (n_total = 실제 로드된 시행 수, NUM_RANDOM_TRIALS 이하일 수 있음).
    """
    cue_row = df_fixed[(df_fixed['Dataset'] == dataset) & (df_fixed['Target'] == target) & (df_fixed['Model'] == 'CUE')]
    df_r = df_random[(df_random['Dataset'] == dataset) & (df_random['Target'] == target)]
    if cue_row.empty or df_r.empty:
        return None

    rows = []
    for m in metrics:
        if m not in cue_row.columns or m not in df_r.columns:
            continue
        cue_val = cue_row[m].values[0]
        trials = df_r[m].dropna().values
        n_total = len(trials)
        if n_total == 0:
            continue
        if LOWER_IS_BETTER.get(m, True):
            n_better = int(np.sum(trials < cue_val))
        else:
            n_better = int(np.sum(trials > cue_val))
        rows.append({
            'metric': m,
            'cue_value': round(float(cue_val), 3),
            'n_better': n_better,
            'n_total': n_total,
            'p_value': round(n_better / n_total, 3),
        })

    return pd.DataFrame(rows) if rows else None


table_save_dir = os.path.join(cue_base, 'table', 'weight_comparison_v2', f'{NUM_RANDOM_TRIALS}')
os.makedirs(table_save_dir, exist_ok=True)

# DIS -> dis(dissociation), INH -> inh(inhibition)
dataset_filenames = {'DIS': 'dis.csv', 'INH': 'inh.csv'}
dataset_pvalue_filenames = {'DIS': 'dis_pvalue.csv', 'INH': 'inh_pvalue.csv'}

# 랭킹표용 통합 데이터: Random은 시행 평균으로 대표한다.
df_random_mean = (
    df_random.groupby(['Dataset', 'Target'], as_index=False)[metrics_of_interest]
    .mean()
)
df_random_mean.insert(2, 'Model', 'Random')
df_total = pd.concat([df_fixed, df_random_mean], ignore_index=True)

ranked_tables = {}
pvalue_tables = {}
for d_name in datasets:
    dataset_tables = []
    dataset_pvalue_tables = []
    for target in targets:
        table = build_ranked_table(df_total, d_name, target)
        pvalue_table = build_pvalue_table(df_random, df_fixed, d_name, target)

        if table is not None:
            ranked_tables[(d_name, target)] = table
            print(f"\n=== [{d_name} / {target}] Ranked ===")
            print(table.to_string(index=False))

            table_with_target = table.copy()
            table_with_target.insert(0, 'Target', target)
            dataset_tables.append(table_with_target)

        if pvalue_table is not None:
            pvalue_tables[(d_name, target)] = pvalue_table
            print(f"\n=== [{d_name} / {target}] p-value (Random better than CUE) ===")
            print(pvalue_table.to_string(index=False))

            pvalue_with_target = pvalue_table.copy()
            pvalue_with_target.insert(0, 'Target', target)
            dataset_pvalue_tables.append(pvalue_with_target)

    if dataset_tables:
        dataset_table = pd.concat(dataset_tables, ignore_index=True)
        save_path = os.path.join(table_save_dir, dataset_filenames[d_name])
        dataset_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
        print(f"\n✅ 저장 완료: {save_path}")

    if dataset_pvalue_tables:
        dataset_pvalue_table = pd.concat(dataset_pvalue_tables, ignore_index=True)
        save_path = os.path.join(table_save_dir, dataset_pvalue_filenames[d_name])
        dataset_pvalue_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
        print(f"✅ 저장 완료: {save_path}")


=== [Kd / PI] Ranked ===
  Model  avg_rank  rmse  rmse_rank    r2  r2_rank   pcc  pcc_rank
     FP       3.3 0.245        3.0 0.098      3.0 0.357       4.0
    IMG       5.0 0.257        5.0 0.008      5.0 0.194       5.0
UNIFORM       2.3 0.242        2.0 0.121      2.0 0.361       3.0
 Random       3.0 0.247        4.0 0.077      4.0 0.368       1.0
    CUE       1.3 0.242        1.0 0.125      1.0 0.366       2.0

=== [Kd / PI] p-value (Random better than CUE) ===
metric  cue_value  n_better  n_total  p_value
  rmse      0.242         3       30    0.100
    r2      0.125         3       30    0.100
   pcc      0.366        17       29    0.586

=== [Kd / Ssel] Ranked ===
  Model  avg_rank  rmse  rmse_rank     r2  r2_rank   pcc  pcc_rank
     FP       1.0 0.887        1.0  0.081      1.0 0.326       1.0
    IMG       5.0 0.957        5.0 -0.072      5.0 0.189       5.0
UNIFORM       3.0 0.901        3.0  0.051      3.0 0.303       3.0
 Random       4.0 0.920        4.0  0.010     